<a href="https://colab.research.google.com/github/24x01a05r9-spec/fullstack-blog-app/blob/main/Blog%20app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request

# Ensure the template structure exists
os.makedirs('templates', exist_ok=True)

# Directly download a highly compressed version of the single-page HTML frontend layout
frontend_code = """<!DOCTYPE html><html><head><meta charset="utf-8"><title>Blog</title><link href="https://jsdelivr.net" rel="stylesheet"></head><body class="bg-light p-4"><div class="container" style="max-width:700px;"><div class="d-flex justify-content-between mb-4"><h2>📝 Colab Blog</h2><div id="ab"></div></div><div id="vb"></div></div><script>const b=window.location.origin;const h=()=>({'Content-Type':'application/json','Authorization':localStorage.getItem('tk')?`Bearer ${localStorage.getItem('tk')}`:''});async function lf(){const u=localStorage.getItem('un');document.getElementById('ab').innerHTML=u?`<span>Hi <b>${u}</b></span><button class="btn btn-sm btn-danger ms-2" onclick="lo()">Logout</button>`:`<button class="btn btn-sm btn-primary" onclick="sl()">Login</button>`;let html=u?`<div class="card p-3 mb-4"><h5>New Post</h5><input id="t" class="form-control mb-2" placeholder="Title"><textarea id="c" class="form-control mb-2" placeholder="Content"></textarea><button class="btn btn-sm btn-success" onclick="ap()">Publish</button></div>`:'';const res=await fetch(`${b}/api/posts`);const posts=await res.json();html+=posts.map(p=>`<div class="card p-3 mb-3"><h4>${p.title}</h4><p class="text-muted small">By ${p.author}</p><p>${p.content}</p><button class="btn btn-link btn-sm p-0 text-start" onclick="lp(${p.id})">View Thread</button></div>`).join('');document.getElementById('vb').innerHTML=html||'<p>No posts yet.</p>';}async function lp(id){const res=await fetch(`${b}/api/posts/${id}`);const p=await res.json();const tk=localStorage.getItem('tk');document.getElementById('vb').innerHTML=`<button class="btn btn-sm btn-secondary mb-3" onclick="lf()">&larr; Back</button><div class="card p-3 mb-3"><h3>${p.title}</h3><p>${p.content}</p></div><div class="card p-3"><h5>Comments</h5>${p.comments.map(c=>`<div class="border-start ps-2 my-2"><b>${c.author}:</b> ${c.content}</div>`).join('')}${tk?`<div class="mt-2"><input id="cc" class="form-control mb-2" placeholder="Write a comment..."><button class="btn btn-sm btn-primary" onclick="ac(${id})">Comment</button></div>`:''}</div>`;}function sl(){document.getElementById('vb').innerHTML=`<div class="card p-3 mx-auto" style="max-width:350px;"><h5>Sign In / Register</h5><input id="le" class="form-control mb-2" placeholder="Email"><input id="lu" class="form-control mb-2" placeholder="Username (Sign up only)"><input id="lp" type="password" class="form-control mb-2" placeholder="Password"><button class="btn btn-sm btn-primary mb-2" onclick="lin()">Sign In</button><button class="btn btn-sm btn-outline-secondary" onclick="reg()">Create Account</button></div>`;}async function lin(){const res=await fetch(`${b}/api/auth/login`,{method:'POST',headers:h(),body:JSON.stringify({email:document.getElementById('le').value,password:document.getElementById('lp').value})});const d=await res.json();if(res.ok){localStorage.setItem('tk',d.token);localStorage.setItem('un',d.username);lf();}else{alert(d.message);}}async function reg(){const res=await fetch(`${b}/api/auth/register`,{method:'POST',headers:h(),body:JSON.stringify({username:document.getElementById('lu').value,email:document.getElementById('le').value,password:document.getElementById('lp').value})});if(res.ok){alert('Account created! Please Sign In.');}else{const d=await res.json();alert(d.message);}}async function ap(){await fetch(`${b}/api/posts`,{method:'POST',headers:h(),body:JSON.stringify({title:document.getElementById('t').value,content:document.getElementById('c').value})});lf();}async function ac(id){await fetch(`${b}/api/posts/${id}/comments`,{method:'POST',headers:h(),body:JSON.stringify({content:document.getElementById('cc').value})});lp(id);}function lo(){localStorage.clear();lf();}lf();</script></body></html>"""

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(frontend_code)

print("✅ Index file written successfully without copy-paste truncation glitches!")

✅ Index file written successfully without copy-paste truncation glitches!


In [2]:
!pip install flask flask-cors pyjwt bcrypt flask-sqlalchemy
!npm install -g localtunnel

import re
import time
import jwt
import bcrypt
import threading
from datetime import datetime, timedelta
from functools import wraps
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__, template_folder='templates')
CORS(app)

app.config['SECRET_KEY'] = 'colab-secure-key'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///blog.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
db = SQLAlchemy(app)

class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    username = db.Column(db.String(50), unique=True, nullable=False)
    email = db.Column(db.String(120), unique=True, nullable=False)
    password = db.Column(db.String(100), nullable=False)

class Post(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(200), nullable=False)
    content = db.Column(db.Text, nullable=False)
    user_id = db.Column(db.Integer, db.ForeignKey('user.id'), nullable=False)
    author_name = db.Column(db.String(50))

class Comment(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    content = db.Column(db.Text, nullable=False)
    user_id = db.Column(db.Integer, db.ForeignKey('user.id'), nullable=False)
    post_id = db.Column(db.Integer, db.ForeignKey('post.id'), nullable=False)
    author_name = db.Column(db.String(50))

with app.app_context():
    db.create_all()

def token_required(f):
    @wraps(f)
    def dec(*args, **kwargs):
        token = request.headers.get('Authorization', '').split(" ")[-1]
        if not token: return jsonify({'message': 'Missing authorization token.'}), 401
        try:
            data = jwt.decode(token, app.config['SECRET_KEY'], algorithms=["HS256"])
            curr = db.session.get(User, data['user_id'])
            if not curr: return jsonify({'message': 'User profile mismatch.'}), 401
        except: return jsonify({'message': 'Session timeout, log back in.'}), 401
        return f(curr, *args, **kwargs)
    return dec

@app.route('/')
def home(): return render_template('index.html')

@app.route('/api/auth/register', methods=['POST'])
def reg():
    d = request.get_json()
    if User.query.filter((User.username==d['username']) | (User.email==d['email'])).first():
        return jsonify({'message': 'Username or Email is already taken.'}), 400
    pwd = bcrypt.hashpw(d['password'].encode('utf-8'), bcrypt.gensalt()).decode('utf-8')
    db.session.add(User(username=d['username'], email=d['email'], password=pwd))
    db.session.commit()
    return jsonify({}), 201

@app.route('/api/auth/login', methods=['POST'])
def log():
    d = request.get_json()
    u = User.query.filter_by(email=d['email']).first()
    if not u or not bcrypt.checkpw(d['password'].encode('utf-8'), u.password.encode('utf-8')):
        return jsonify({'message': 'Incorrect email or password.'}), 401
    tk = jwt.encode({'user_id': u.id, 'exp': datetime.utcnow() + timedelta(hours=24)}, app.config['SECRET_KEY'], algorithm="HS256")
    return jsonify({'token': tk, 'username': u.username}), 200

@app.route('/api/posts', methods=['GET', 'POST'])
def posts_handler():
    if request.method == 'POST':
        @token_required
        def create(curr):
            d = request.get_json()
            db.session.add(Post(title=d['title'], content=d['content'], user_id=curr.id, author_name=curr.username))
            db.session.commit()
            return jsonify({}), 201
        return create()
    all_p = Post.query.all()
    return jsonify([{'id':p.id, 'title':p.title, 'content':p.content, 'author':p.author_name} for p in all_p]), 200

@app.route('/api/posts/<int:pid>', methods=['GET'])
def get_p(pid):
    p = db.session.get(Post, pid)
    coms = Comment.query.filter_by(post_id=pid).all()
    return jsonify({'title':p.title, 'content':p.content, 'comments':[{'content':c.content, 'author':c.author_name} for c in coms]}), 200

@app.route('/api/posts/<int:pid>/comments', methods=['POST'])
@token_required
def add_c(curr, pid):
    d = request.get_json()
    db.session.add(Comment(content=d['content'], user_id=curr.id, post_id=pid, author_name=curr.username))
    db.session.commit()
    return jsonify({}), 201

threading.Thread(target=lambda: app.run(port=5000, host='0.0.0.0', use_reloader=False), daemon=True).start()
time.sleep(2)

import urllib.request
ip = urllib.request.urlopen('https://icanhazip.com').read().decode('utf8').strip()
print(f"\n" + "="*50 + f"\n🔑 ENDPOINT IP FOR SECURITY CHECK: {ip}\n" + "="*50 + "\n")

get_ipython().system_raw('lt --port 5000 > lt.log 2>&1 &')
time.sleep(3)
with open('lt.log', 'r') as f:
    urls = re.findall(r'https://[a-zA-Z0-9-]+\.loca\.lt', f.read())
    if urls: print(f"👉 CLICK TO LAUNCH BLOG: {urls}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.4 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋ * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



🔑 ENDPOINT IP FOR SECURITY CHECK: 35.253.28.241

👉 CLICK TO LAUNCH BLOG: ['https://green-pears-report.loca.lt']
